# LangCalc evaluation

<img src="https://raw.githubusercontent.com/jedick/LangCalc/main/assets/langcalc-icon-outline.svg" alt="LangCalc icon" width="100">

This notebook evaluates function-calling models (base and fine-tuned FunctionGemma and a larger general-purpose model) on the test set of [LangCalc](https://github.com/jedick/LangCalc) spoken calculator requests.

<a target="_blank" href="https://colab.research.google.com/github/jedick/LangCalc/blob/main/model/notebooks/LangCalc-evaluation.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>

<a target="_blank" href="https://github.com/jedick/LangCalc/blob/main/model/notebooks/LangCalc-evaluation.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>

## How to use this notebook

1. Run all the function-definition cells below once. They are collapsed by default; expand any cell to see what it does.
2. Edit the parameters in the **main cell** at the bottom: which models to evaluate, which dataset each one uses, and how many replicates to run.
3. Run the main cell. Progress bars report the current model, replicate, and a running count of correct responses.
4. A results table, a horizontal bar chart, a Markdown report (`test-results.md` by default), and a per-model results CSV (`test-results_<model_id>.csv`, with the test data plus each replicate's raw output) are saved to the notebook's working directory, ready to drop into the [LangCalc repo](https://github.com/jedick/LangCalc) README or a dedicated results page.

Only the function call and its arguments are scored here (e.g. `multiply(2, 20)`), not the arithmetic result, since the arithmetic itself is plain deterministic code rather than something the model needs to get right.

**Runtime:** generation itself takes roughly 1.5 minutes per replicate (100 test prompts). The first time you evaluate a given model, expect several extra minutes to download its weights (around 10 GB for Gemma 4 E2B-it).

## Function definitions

Run them all once before the main cell.

### Install dependencies

In [ ]:
# Install dependencies (safe to skip if already installed in your environment)
# NOTE: accelerate is required for device_map=auto
%pip install -q transformers accelerate datasets torch tqdm matplotlib pandas huggingface_hub

### Hugging Face login

In [ ]:
# Logging in may raise your Hugging Face rate limits and is required to access
# the gated FunctionGemma base model (but not the fine-tuned model or Gemma 4).
# Controlled by the `use_hf_auth` parameter in the main cell.

def hf_login():
    """Log in to the Hugging Face Hub, using a Colab secret if available,
    otherwise falling back to an interactive prompt."""
    from huggingface_hub import login

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass

    if token:
        login(token)
        print("Logged in to Hugging Face Hub using the HF_TOKEN Colab secret.")
    else:
        login()

### Tool definitions

In [ ]:
# Same functions used for training and inference (see LangCalc-inference.ipynb
# and langcalc_gemma4.py). Passed as raw Python functions to
# apply_chat_template(tools=...), which both model families accept.

def add(x: float, y: float):
    """
    Adds two numbers together (sum, total, plus).

    Args:
        x: the first number
        y: the second number

    Returns:
        result: the sum of x and y (x + y).
    """
    return {"result": x + y}


def subtract(x: float, y: float):
    """
    Subtracts one number from another (difference, minus).

    Args:
        x: the starting number
        y: the number to be subtracted from x

    Returns:
        result: the difference between x and y (x - y).
    """
    return {"result": x - y}


def multiply(x: float, y: float):
    """
    Multiplies two numbers together (product, times).

    Args:
        x: the first number
        y: the second number

    Returns:
        result: the product of x and y (x * y).
    """
    return {"result": x * y}


def divide(x: float, y: float):
    """
    Divides one number by another (quotient, over).

    Args:
        x: the numerator
        y: the denominator

    Returns:
        result: the quotient of x divided by y (x / y).
    """
    return {"result": x / y}

### Model family adapters

In [ ]:
# The two model families we support use different chat templates and
# different tags for tool calls:
#   - "functiongemma": FunctionGemma-style models (e.g. the fine-tuned
#     jedick/functiongemma-langcalc-en). Uses a "developer" role
#     message, <start_function_call>/<end_function_call> tags, and
#     <escape> for quoting string arguments. See LangCalc-inference.ipynb.
#   - "gemma4": Gemma 4 models (e.g. google/gemma-4-E2B). Uses a "system"
#     role message, <|tool_call>/<tool_call|> tags, and <|"|> for quoting
#     string arguments, and attaches "tool_calls" and "tool_responses" to a
#     single assistant message instead of a separate "tool" role message.
#     See langcalc_gemma4.py.
#
# detect_model_family() guesses the family from the model id (currently:
# anything with "gemma-4" in the id is treated as "gemma4", everything else
# as "functiongemma"). If you evaluate a model that is neither, add a new
# family entry to FAMILY_CONFIGS so its tags and message format match its
# own chat template.

import re


def detect_model_family(model_id):
    return "gemma4" if "gemma-4" in model_id.lower() else "functiongemma"


def cast_value(v):
    try:
        return int(v)
    except ValueError:
        try:
            return float(v)
        except ValueError:
            return {"true": True, "false": False}.get(v.lower(), v.strip("'\""))


def extract_tool_calls_functiongemma(text):
    return [
        {
            "name": name,
            "arguments": {
                k: cast_value((v1 or v2).strip())
                for k, v1, v2 in re.findall(r"(\w+):(?:<escape>(.*?)<escape>|([^,}]*))", args)
            },
        }
        for name, args in re.findall(
            r"<start_function_call>call:(\w+)\{(.*?)\}<end_function_call>", text, re.DOTALL
        )
    ]


def extract_tool_calls_gemma4(text):
    return [
        {
            "name": name,
            "arguments": {
                k: cast_value((v1 or v2).strip())
                for k, v1, v2 in re.findall(r'(\w+):(?:<\|"\|>(.*?)<\|"\|>|([^,}]*))', args)
            },
        }
        for name, args in re.findall(
            r"<\|tool_call>call:(\w+)\{(.*?)\}<tool_call\|>", text, re.DOTALL
        )
    ]


def prepare_inputs_functiongemma(processor, messages, tools, model):
    inputs = processor.apply_chat_template(
        messages, tools=tools, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    )
    return inputs.to(model.device)


def prepare_inputs_gemma4(processor, messages, tools, model):
    text = processor.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=text, return_tensors="pt")
    return inputs.to(model.device)


FAMILY_CONFIGS = {
    "functiongemma": {
        "system_role": "developer",
        "system_message": "You are a model that can do function calling with the following functions",
        "prepare_inputs": prepare_inputs_functiongemma,
        "extract_tool_calls": extract_tool_calls_functiongemma,
        # FunctionGemma's function-call tags survive skip_special_tokens=True
        # (they aren't registered as special tokens), so we can decode cleanly.
        "decode_kwargs": {"skip_special_tokens": True},
        "generate_kwargs": lambda processor: {"pad_token_id": processor.eos_token_id},
    },
    "gemma4": {
        "system_role": "system",
        "system_message": "You are a model that can do function calling for a calculator app.",
        "prepare_inputs": prepare_inputs_gemma4,
        "extract_tool_calls": extract_tool_calls_gemma4,
        # Gemma 4's tool-call tags ARE special tokens, so we need
        # skip_special_tokens=False or the tags get stripped before we can
        # parse them.
        "decode_kwargs": {"skip_special_tokens": False},
        "generate_kwargs": lambda processor: {},
    },
}

### Dataset loading

In [ ]:
# Expects a Hugging Face dataset repo with (at least) these columns:
# prompt, function, x, y.

from datasets import load_dataset

_dataset_cache = {}


def load_eval_examples(repo_id, split):
    """Load one split of a HF dataset repo, caching by (repo_id, split) so
    the same dataset isn't downloaded again for a second model that uses
    the same evaluation set."""
    key = (repo_id, split)
    if key not in _dataset_cache:
        raw = load_dataset(repo_id)
        if split not in raw:
            raise ValueError(
                f"Split '{split}' not found in dataset '{repo_id}'. "
                f"Available splits: {list(raw.keys())}"
            )
        _dataset_cache[key] = raw[split]
    return _dataset_cache[key]

### Model loading

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

try:
    from transformers import AutoModelForMultimodalLM
except ImportError:
    AutoModelForMultimodalLM = None


def load_model_and_processor(model_id, family):
    """Load a model and its processor, picking the model class each family
    expects (see langcalc_gemma4.py for Gemma 4). Falls back to
    AutoModelForCausalLM if AutoModelForMultimodalLM isn't available in the
    installed transformers version."""
    processor = AutoProcessor.from_pretrained(model_id)

    model_class = AutoModelForCausalLM
    if family == "gemma4" and AutoModelForMultimodalLM is not None:
        model_class = AutoModelForMultimodalLM
    elif family == "gemma4":
        print("AutoModelForMultimodalLM is not available in this transformers "
              "version; falling back to AutoModelForCausalLM.")

    device_map = "auto" if torch.cuda.is_available() else None
    model = model_class.from_pretrained(model_id, dtype="auto", device_map=device_map)
    if device_map is None:
        model = model.to("cpu")
    model.eval()
    return model, processor

### Inference and scoring

In [ ]:
# We only need the model's first turn (the function call itself), not the
# follow-up "here is the arithmetic result" turn, so each example needs
# exactly one generation call.

import math


def run_inference(model, processor, family, tools, prompt, max_new_tokens=128):
    cfg = FAMILY_CONFIGS[family]
    messages = [
        {"role": cfg["system_role"], "content": cfg["system_message"]},
        {"role": "user", "content": prompt},
    ]
    inputs = cfg["prepare_inputs"](processor, messages, tools, model)

    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, **cfg["generate_kwargs"](processor)
        )

    generated_tokens = out[0][len(inputs["input_ids"][0]):]
    text = processor.decode(generated_tokens, **cfg["decode_kwargs"])
    calls = cfg["extract_tool_calls"](text)
    return calls[0] if calls else None


def numbers_match(actual, expected, tol=1e-6):
    if actual is None:
        return False
    try:
        return math.isclose(float(actual), float(expected), rel_tol=tol, abs_tol=tol)
    except (TypeError, ValueError):
        return False


# add and multiply are commutative, so either argument order counts as correct
COMMUTATIVE_FUNCTIONS = {"add", "multiply"}


def args_match(call_args, expected_x, expected_y, expected_function):
    actual_x, actual_y = call_args.get("x"), call_args.get("y")
    if numbers_match(actual_x, expected_x) and numbers_match(actual_y, expected_y):
        return True
    if expected_function in COMMUTATIVE_FUNCTIONS:
        return numbers_match(actual_x, expected_y) and numbers_match(actual_y, expected_x)
    return False


def score_call(call, example):
    """Compare one model response to its expected function call. Returns
    one of four mutually exclusive categories."""
    expected_function = example["function"]
    expected_x, expected_y = float(example["x"]), float(example["y"])

    name_correct = call is not None and call["name"] == expected_function
    args_correct = call is not None and args_match(
        call["arguments"], expected_x, expected_y, expected_function
    )

    if name_correct and args_correct:
        return "correct"
    if not name_correct and args_correct:
        return "wrong_function"
    if name_correct and not args_correct:
        return "wrong_arguments"
    return "wrong_both"


CATEGORIES = ["correct", "wrong_function", "wrong_arguments", "wrong_both"]
CATEGORY_LABELS = {
    "correct": "Correct",
    "wrong_function": "Wrong function only",
    "wrong_arguments": "Wrong arguments only",
    "wrong_both": "Wrong function and arguments",
}

### One evaluation replicate over a dataset

In [ ]:
from tqdm.auto import tqdm


def evaluate_replicate(model, processor, family, tools, examples, model_label,
                        replicate, n_replicates, max_new_tokens=128):
    """Returns (counts, calls): counts is the per-category tally, calls is the
    raw model output (or None) for each example, in order, for CSV export."""
    counts = {c: 0 for c in CATEGORIES}
    calls = []
    desc = f"{model_label} (replicate {replicate}/{n_replicates})"
    with tqdm(examples, desc=desc) as pbar:
        for example in pbar:
            call = run_inference(
                model, processor, family, tools, example["prompt"], max_new_tokens=max_new_tokens
            )
            calls.append(call)
            category = score_call(call, example)
            counts[category] += 1
            pbar.set_postfix(done=sum(counts.values()), correct=counts["correct"])
    return counts, calls

### Per-example CSV export

In [ ]:
# One CSV per model: the test data (category, prompt, function, x, y) plus
# each replicate's raw output (function_1, x_1, y_1, function_2, x_2, y_2, ...).

import os
import re as _re
import pandas as pd


def sanitize_model_id(model_id):
    return _re.sub(r"[^A-Za-z0-9._-]+", "__", model_id)


def write_results_csv(model_id, examples, replicate_calls, csv_dir="."):
    n_replicates = len(replicate_calls)
    rows = []
    for idx, example in enumerate(examples):
        row = {
            "category": example.get("category"),
            "prompt": example["prompt"],
            "function": example["function"],
            "x": example["x"],
            "y": example["y"],
        }
        for r in range(n_replicates):
            call = replicate_calls[r][idx]
            row[f"function_{r + 1}"] = call["name"] if call else None
            row[f"x_{r + 1}"] = call["arguments"].get("x") if call else None
            row[f"y_{r + 1}"] = call["arguments"].get("y") if call else None
        rows.append(row)

    df = pd.DataFrame(rows)
    os.makedirs(csv_dir, exist_ok=True)
    csv_path = os.path.join(csv_dir, f"test-results_{sanitize_model_id(model_id)}.csv")
    df.to_csv(csv_path, index=False)
    return csv_path

### Full evaluation for one model: load once, run several replicates

In [ ]:
import gc
import statistics


def evaluate_model(model_id, dataset_id, split, tools, n_replicates=3, max_new_tokens=128,
                    csv_dir="."):
    family = detect_model_family(model_id)
    print(f"\n=== {model_id} (family: {family}) ===")

    examples = load_eval_examples(dataset_id, split)
    print(f"Evaluation set: {dataset_id} [{split}], {len(examples)} prompts, {n_replicates} replicates")

    model, processor = load_model_and_processor(model_id, family)

    replicate_results = [
        evaluate_replicate(
            model, processor, family, tools, examples,
            model_label=model_id, replicate=i + 1, n_replicates=n_replicates,
            max_new_tokens=max_new_tokens,
        )
        for i in range(n_replicates)
    ]
    replicate_counts = [counts for counts, _ in replicate_results]
    replicate_calls = [calls for _, calls in replicate_results]

    # Free GPU memory before loading the next model
    del model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    csv_path = write_results_csv(model_id, examples, replicate_calls, csv_dir)
    print(f"Saved per-example results to {csv_path}")

    n = len(examples)
    percentages = {c: [100 * rc[c] / n for rc in replicate_counts] for c in CATEGORIES}
    summary = {
        "model_id": model_id,
        "dataset": f"{dataset_id} [{split}]",
        "n_prompts": n,
        "n_replicates": n_replicates,
    }
    for c in CATEGORIES:
        summary[f"{c}_mean"] = statistics.mean(percentages[c])
        summary[f"{c}_std"] = statistics.stdev(percentages[c]) if n_replicates > 1 else 0.0

    return summary

### Results table

In [ ]:
import pandas as pd


def build_summary_table(summaries):
    rows = []
    for s in summaries:
        row = {"Model": s["model_id"]}
        for c in CATEGORIES:
            row[CATEGORY_LABELS[c]] = f"{s[f'{c}_mean']:.1f} ± {s[f'{c}_std']:.1f}"
        rows.append(row)
    return pd.DataFrame(rows)


def dataframe_to_markdown(df):
    """Render a DataFrame as a Markdown table without requiring the optional
    `tabulate` dependency."""
    header = "| " + " | ".join(df.columns) + " |"
    separator = "| " + " | ".join("---" for _ in df.columns) + " |"
    body = "\n".join(
        "| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)
    )
    return "\n".join([header, separator, body])

### Horizontal stacked bar chart

In [ ]:
# One bar per model, split into the four scoring categories, so the chart is
# readable on its own (with a legend) if dropped into a README.

import matplotlib.pyplot as plt

CATEGORY_COLORS = {
    "correct": "#63B6FF",
    "wrong_function": "#FFC107",
    "wrong_arguments": "#FF9800",
    "wrong_both": "#F44336",
}


def build_results_chart(summaries, dataset_id, split, results_dir):
    models = [s["model_id"] for s in summaries]
    fig, ax = plt.subplots(figsize=(8, 0.9 * len(models) + 1.5))

    left = [0] * len(models)
    for c in CATEGORIES:
        values = [s[f"{c}_mean"] for s in summaries]
        ax.barh(models, values, left=left, color=CATEGORY_COLORS[c], label=CATEGORY_LABELS[c])
        for i, (v, l) in enumerate(zip(values, left)):
            if v >= 5:
                ax.text(l + v / 2, i, f"{v:.0f}%", ha="center", va="center", fontsize=9, color="black")
        left = [l + v for l, v in zip(left, values)]

    ax.set_xlim(0, 100)
    ax.set_xlabel("Percentage of test prompts")

    # Parse a language abbreviation from a dataset repo name (e.g. 'jedick/langcalc-en' -> 'en')
    language_label = dataset_id.rsplit("-", 1)[-1]
    chart_title = f"Function-calling accuracy on LangCalc ({language_label}) {split} data"
    ax.set_title(chart_title)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
    plt.tight_layout()
    save_path = os.path.join(results_dir, "test-results.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved chart to {save_path}")

### Markdown report

In [ ]:
def write_markdown_report(summaries, table_markdown, results_dir):
    n_prompts = summaries[0]["n_prompts"] if summaries else "?"
    n_replicates = summaries[0]["n_replicates"] if summaries else "?"
    lines = [
        "# LangCalc evaluation results",
        "",
        f"Mean ± standard deviation over {n_replicates} replicates, "
        f"{n_prompts} test prompts per replicate. Values are percentages of test prompts.",
        "",
        table_markdown,
        "",
        "![LangCalc evaluation results](test-results.png)",
        "",
    ]
    content = "\n".join(lines)
    report_path = os.path.join(results_dir, "test-results.md")
    with open(report_path, "w") as f:
        f.write(content)
    print(f"Saved report to {report_path}")

## Run evaluation

Set the parameters below, then run this cell. See **How to use this notebook** above for what happens and how long it takes.

In [ ]:
# ============================================================
# MAIN — set parameters below, then run this cell
# ============================================================

use_hf_auth = True  # @param {type:"boolean"}
n_replicates = 3  # @param {type:"integer"}
max_new_tokens = 128  # @param {type:"integer"}

dataset_id = "jedick/langcalc-en"  # @param {type:"string"}
split = "test"  # @param {type:"string"}
results_dir = "."  # @param {type:"string"}

# Up to three models to compare. Leave model_3_id blank to evaluate only two.
model_1_id = "google/functiongemma-270m-it"  # @param {type:"string"}
model_2_id = "jedick/functiongemma-langcalc-en"  # @param {type:"string"}
model_3_id = "jedick/functiongemma-langcalc-en.zh"  # @param {type:"string"}

# ------------------------------------------------------------

if use_hf_auth:
    hf_login()

tools = [add, subtract, multiply, divide]

model_configs = [model_1_id, model_2_id, model_3_id]
model_configs = [m for m in model_configs if m.strip()]  # drop empty slot(s)

if not (1 <= len(model_configs) <= 3):
    raise ValueError("Fill in between 1 and 3 model ids to evaluate.")

os.makedirs(results_dir, exist_ok=True)

summaries = [
    evaluate_model(
        model_id, dataset_id, split, tools,
        n_replicates=n_replicates, max_new_tokens=max_new_tokens,
        csv_dir=results_dir,
    )
    for model_id in model_configs
]

summary_table = build_summary_table(summaries)
table_markdown = dataframe_to_markdown(summary_table)
print("\n" + table_markdown)

build_results_chart(summaries, dataset_id, split, results_dir)
write_markdown_report(summaries, table_markdown, results_dir)